# 第14讲 · 机器学习入门 —— 线性回归

本 notebook 对应课件 `lecture14.md` 与脚本 `linear_regression.py`。

**一句话目标**：找到一条直线 `y = kx + b`，让所有数据点到它的距离之和最小，从而用 `x` 预测 `y`。

完整流程：
1. 面积-价格数据 → `LinearRegression` 拟合（读系数 / 截距）
2. 预测新样本并核对手写公式 ŷ = k·x + b
3. 散点 + 拟合线可视化
4. 进阶：温度-能耗
5. 挑战：离群点 / 多特征

依赖：`numpy`、`scikit-learn`、`matplotlib`。

In [ ]:
# 让绘图直接嵌入 notebook（不弹窗）
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from plots import setup_zh_font

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 中文字体设置
setup_zh_font()

## 手写公式核对工具

`manual_predict`：用手写公式 `ŷ = b + k·x` 核对 `model.predict`，验证 `sklearn` 内部找到的正是这条直线。

In [ ]:
def manual_predict(model, x):
    """用手写公式 ŷ = b + k·x 核对 model.predict。"""
    return float(model.intercept_ + model.coef_[0] * x)

## 1. 面积-价格拟合

房屋面积（㎡）→ 价格（万元）。`LinearRegression().fit` 内部自动做**最小二乘**求最优 `k`、`b`：
- `coef_`：斜率 k
- `intercept_`：截距 b

In [ ]:
area = np.array([50, 70, 80, 100, 120, 130, 150]).reshape(-1, 1)
price = np.array([45, 62, 70, 88, 105, 112, 130])

model = LinearRegression()
model.fit(area, price)

print("=== 面积-价格 线性回归 ===")
print(f"斜率 k (coef_) = {model.coef_[0]:.4f}")
print(f"截距 b (intercept_) = {model.intercept_:.4f}")

## 2. 预测新样本 + 与手写公式核对

预测 95㎡ 的价格，同时用手写公式 `ŷ = b + k·x` 验证两者一致。

In [ ]:
new_area = np.array([[95]])
pred = model.predict(new_area)
print(f"95㎡ 预测价格 = {pred[0]:.2f} 万元")
manual = manual_predict(model, 95)
print(f"手写公式 ŷ=k·x+b = {manual:.2f} 万元  （与 predict 一致 ✓）")

## 3. 散点 + 拟合线可视化

画出原始数据点，再叠上拟合直线，直观看到直线穿过了数据的整体趋势。

In [ ]:
def plot_fit(x, y, model, title, xlabel, ylabel):
    """散点 + 拟合直线（内嵌显示）。"""
    xs = np.linspace(x.min() - 5, x.max() + 5, 100)
    ys = model.intercept_ + model.coef_[0] * xs
    plt.figure(figsize=(6.4, 4.8))
    plt.scatter(x, y, color="blue", label="原始数据")
    plt.plot(xs, ys, color="red", label="拟合直线 y=kx+b")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.4)
    plt.show()

plot_fit(area, price, model, "房屋面积-价格 线性回归",
         "面积 (㎡)", "价格 (万元)")

## 4. 进阶：温度-能耗

温度越高，能耗先降后升（空调/取暖切换），再用线性回归只能拟合整体直线趋势。预测 18℃ 的能耗。

In [ ]:
temp = np.array([5, 10, 15, 20, 25, 30, 35]).reshape(-1, 1)
energy = np.array([9, 7, 5.8, 5, 5.5, 7.2, 10])
m2 = LinearRegression().fit(temp, energy)
print("=== 进阶：温度-能耗 ===")
print(f"斜率 k = {m2.coef_[0]:.4f}, 截距 b = {m2.intercept_:.4f}")
t_new = np.array([[18]])
print(f"18℃ 预测能耗 = {m2.predict(t_new)[0]:.2f} （手写核对 = "
      f"{manual_predict(m2, 18):.2f}）")

## 5. 挑战：加入离群点

往面积-价格数据里加一个离群点（155㎡，300万元），看斜率如何变化。

> **最小二乘对离群点敏感**——离群点把它那侧的直线往上拽，斜率明显变大。

In [ ]:
area_out = np.array([50, 70, 80, 100, 120, 130, 150, 155]).reshape(-1, 1)
price_out = np.array([45, 62, 70, 88, 105, 112, 130, 300])  # 离群点 300
m3 = LinearRegression().fit(area_out, price_out)
print("=== 挑战：加入离群点(155㎡,300万) ===")
print(f"斜率由 {model.coef_[0]:.3f} → {m3.coef_[0]:.3f}")
print("  离群点把拟合线往上拽，说明最小二乘对离群点敏感。")

### 离群点前后拟合线对比

In [ ]:
xs = np.linspace(area_out.min() - 5, area_out.max() + 5, 100)

plt.figure(figsize=(6.4, 4.8))
plt.scatter(area, price, color="blue", label="原始数据")
plt.scatter([155], [300], color="orange", s=80, label="离群点(155,300)")
plt.plot(xs, model.intercept_ + model.coef_[0] * xs, "b-", label="无离群点拟合")
plt.plot(xs, m3.intercept_ + m3.coef_[0] * xs, "r--", label="含离群点拟合")
plt.xlabel("面积 (㎡)"); plt.ylabel("价格 (万元)")
plt.title("离群点对线性回归拟合线的影响")
plt.legend(); plt.grid(alpha=0.4)
plt.show()

## 小结

- `LinearRegression` 用最小二乘求最优 `k`、`b`，`coef_` / `intercept_` 可读出。
- 手写公式 `ŷ = b + k·x` 与 `predict` 一致，理解内部机制。
- 最小二乘对离群点敏感，实际应用中常需先清洗异常值或改用更稳健的回归。